In [1]:
import json
import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/processed/clean_sessions.csv", parse_dates=["StartDatetime", "EndDatetime"])

# History features only make sense if rows are in time order
df = df.sort_values("StartDatetime").reset_index(drop=True)
print(df.shape)

(72826, 14)


In [2]:
df["Hour"] = df["StartDatetime"].dt.hour
df["Day"] = df["StartDatetime"].dt.day                 # day of the month
df["Month"] = df["StartDatetime"].dt.month
df["DayOfWeek"] = df["StartDatetime"].dt.dayofweek     # Monday = 0 ... Sunday = 6
df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype(int)   # 1 = Sat/Sun, 0 = weekday

# Group hours into named parts of the day
df["TimeOfDay"] = pd.cut(
    df["Hour"],
    bins=[-1, 5, 11, 16, 20, 23],
    labels=["Night", "Morning", "Afternoon", "Evening", "LateEvening"],
).astype(str)

print(df["TimeOfDay"].value_counts())

TimeOfDay
Evening        23125
Afternoon      20877
Morning        18187
LateEvening     8196
Night           2441
Name: count, dtype: int64


C:\Users\yoges\AppData\Local\Temp\ipykernel_12816\293687691.py:4: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["DayOfWeek"] = df["StartDatetime"].dt.dayofweek     # Monday = 0 ... Sunday = 6


In [3]:
# For every session, look ONLY at what happened BEFORE it.
by_charger = df.groupby("ChargerID")

# cumcount() numbers each charger's sessions 0, 1, 2... = how many came before
df["ChargerPrevSessions"] = by_charger.cumcount()

# shift() moves values down one row, so a session never sees its own demand.
# expanding().mean() then averages everything above it.
df["ChargerPrevAvgDemand"] = by_charger["Demand"].transform(
    lambda s: s.shift().expanding().mean()
)

# Same idea for users, but UserID 0 means "unknown", so use known users only
known = df[df["IsKnownUser"]].copy()
by_user = known.groupby("UserID")
known["UserPrevSessions"] = by_user.cumcount()
known["UserPrevAvgDemand"] = by_user["Demand"].transform(
    lambda s: s.shift().expanding().mean()
)

df = df.join(known[["UserPrevSessions", "UserPrevAvgDemand"]])
df["UserPrevSessions"] = df["UserPrevSessions"].fillna(0)

print("No charger history yet:", df["ChargerPrevAvgDemand"].isna().mean().round(3))   # about 0.029
print("No user history yet:", df["UserPrevAvgDemand"].isna().mean().round(3))         # about 0.465

No charger history yet: 0.029
No user history yet: 0.465


In [4]:
busiest_charger = df["ChargerID"].value_counts().index[0]
example = df[df["ChargerID"] == busiest_charger].head(5)
display(example[["StartDatetime", "Demand", "ChargerPrevSessions", "ChargerPrevAvgDemand"]])

# The 3rd session's history should equal the average of sessions 1 and 2 only
manual_average = example["Demand"].iloc[:2].mean()
print("Manual average of first two sessions:", round(manual_average, 3))
print("Value stored for third session:      ", round(example["ChargerPrevAvgDemand"].iloc[2], 3))

,StartDatetime,Demand,ChargerPrevSessions,ChargerPrevAvgDemand
131,2021-10-01 18:58:21,6.36,0,NaN
192,2021-10-02 11:32:41,8.57,1,6.360000
338,2021-10-03 14:05:05,7.43,2,7.465000
363,2021-10-03 17:40:59,4.38,3,7.453333
371,2021-10-03 18:57:00,9.52,4,6.685000


Manual average of first two sessions: 7.465
Value stored for third session:       7.465


In [5]:
previous_end = df.groupby("ChargerID")["EndDatetime"].shift()
still_running = (previous_end > df["StartDatetime"]).sum()
print("Sessions that start while the charger's previous session is unfinished:", still_running)   # about 80

Sessions that start while the charger's previous session is unfinished: 80


In [6]:
# Models need numbers. One-hot encoding makes one 0/1 column per category.
location_dummies = pd.get_dummies(df["Location"].str.replace(" ", "_"), prefix="Loc", dtype=int)
timeofday_dummies = pd.get_dummies(df["TimeOfDay"], prefix="TOD", dtype=int)

df = pd.concat([df, location_dummies, timeofday_dummies], axis=1)
print("Location columns:", location_dummies.shape[1], "| TimeOfDay columns:", timeofday_dummies.shape[1])   # 14 and 5

Location columns: 14 | TimeOfDay columns: 5


In [7]:
cutoff = df["StartDatetime"].quantile(0.8)
train = df[df["StartDatetime"] < cutoff].copy()
test = df[df["StartDatetime"] >= cutoff].copy()

print("Cutoff:", cutoff)                                  # 2022-07-30 19:09:37
print("Train:", len(train), "| Test:", len(test))         # 58,260 and 14,566
print("Train period:", train["StartDatetime"].min().date(), "to", train["StartDatetime"].max().date())
print("Test period: ", test["StartDatetime"].min().date(), "to", test["StartDatetime"].max().date())

Cutoff: 2022-07-30 19:09:37
Train: 58260 | Test: 14566
Train period: 2021-09-30 to 2022-07-30
Test period:  2022-07-30 to 2022-09-30


In [8]:
# High Demand = Demand >= median. We take the median from TRAINING data only.
median_demand = train["Demand"].median()
train["HighDemand"] = (train["Demand"] >= median_demand).astype(int)
test["HighDemand"] = (test["Demand"] >= median_demand).astype(int)

print("Median used:", median_demand)                                         # 14.2
print("High share  train:", round(train["HighDemand"].mean(), 3),
      "| test:", round(test["HighDemand"].mean(), 3))                         # 0.500 and 0.488

# Missing history: flag it, then fill with the TRAINING mean demand
fill_value = train["Demand"].mean()
for data in (train, test):
    data["HasChargerHistory"] = (data["ChargerPrevSessions"] > 0).astype(int)
    data["HasUserHistory"] = (data["UserPrevSessions"] > 0).astype(int)
    data["ChargerPrevAvgDemand"] = data["ChargerPrevAvgDemand"].fillna(fill_value)
    data["UserPrevAvgDemand"] = data["UserPrevAvgDemand"].fillna(fill_value)

Median used: 14.2
High share  train: 0.5 | test: 0.488


In [9]:
base_features = ["ChargerCompany", "ChargerType", "Hour", "Day", "Month", "DayOfWeek", "IsWeekend"]
history_features = [
    "ChargerPrevSessions", "ChargerPrevAvgDemand", "HasChargerHistory",
    "UserPrevSessions", "UserPrevAvgDemand", "HasUserHistory",
]
dummy_features = list(location_dummies.columns) + list(timeofday_dummies.columns)

FEATURES = base_features + history_features + dummy_features
print("Number of features:", len(FEATURES))      # 32

# Columns that must NEVER be features (they leak the answer or are not usable)
forbidden = ["Demand", "HighDemand", "Duration", "DurationCalc", "DurationMismatch",
             "StartDatetime", "EndDatetime", "UserID", "ChargerID"]
assert not set(forbidden) & set(FEATURES), "a leaking column is in FEATURES"

Number of features: 32


In [10]:
assert train["StartDatetime"].max() < test["StartDatetime"].min(), "train and test overlap in time"
assert train[FEATURES].isna().sum().sum() == 0, "missing values in train features"
assert test[FEATURES].isna().sum().sum() == 0, "missing values in test features"
print("All checks passed")

All checks passed


In [11]:
train.to_csv("../data/processed/train.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

with open("../models/feature_columns.json", "w") as file:
    json.dump({"features": FEATURES, "high_demand_threshold": float(median_demand),
               "history_fill_value": float(fill_value)}, file, indent=2)

print("Saved train, test and feature list")

Saved train, test and feature list
